## 1. Load Processed Data

In [2]:
import pandas as pd
fraud_df = pd.read_csv("../data/processed/fraud_with_country.csv")

In [4]:
X = fraud_df.drop(
    columns=[
        "class",
        "signup_time",
        "purchase_time",
        "user_id",
        "device_id",
        "ip_address"
    ]
)

y = fraud_df["class"]

In [5]:
X = pd.get_dummies(
    X,
    columns=[
        "source",
        "browser",
        "sex",
        "country"
    ],
    drop_first=True
)

## 2. Train-Test Split

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [7]:
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

class
0    0.906373
1    0.093627
Name: proportion, dtype: float64
class
0    0.906366
1    0.093634
Name: proportion, dtype: float64


## 3. Handle Class Imbalance

In [ ]:
import sys, subprocess

try:
    from imblearn.over_sampling import SMOTE
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "imbalanced-learn"])
    from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

ModuleNotFoundError: No module named 'imblearn'

In [ ]:
print(y_train.value_counts())
print(y_train_smote.value_counts())

## 4. Logistic Regression Baseline

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000, random_state=42)

lr.fit(X_train_smote, y_train_smote)

ValueError: could not convert string to float: '2015-01-25 03:54:52'

In [ ]:
y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1]

## 5. Evaluate Logistic Regression

In [ ]:
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    confusion_matrix
)

auc_pr_lr = average_precision_score(
    y_test,
    y_prob_lr
)

f1_lr = f1_score(
    y_test,
    y_pred_lr
)

cm_lr = confusion_matrix(
    y_test,
    y_pred_lr
)

print("AUC-PR:", auc_pr_lr)
print("F1:", f1_lr)
print(cm_lr)

## 6. Buid Ensemble Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(
    X_train_smote,
    y_train_smote
)

In [ ]:
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:,1]

## 7. Random Forest Model

In [ ]:
auc_pr_rf = average_precision_score(y_test, y_prob_rf)

f1_rf_rf = f1_score(y_test, y_pred_rf)

cm_rf = confusion_matrix(y_test, y_pred_rf)

print("AUC-PR:", auc_pr_rf)
print("F1 Score:", f1_rf_rf)
print(cm_rf)

## 8. Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

params = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, 15]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42), 
    param_grid=params,
    scoring="f1",
    cv=5,
    n_jobs=-1
)


grid.fit(X_train_smote, y_train_smote)

print(grid.best_params_)

In [ ]:
best_rf = grid.best_estimator_

## 9. Cross Validation

In [ ]:
from sklearn.model_selection import stratifiedKFold
from sklearn.model_selection import cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    best_rf,
    x,
    y,
    cv=cv,
    scoring="f1"
)

print("Mean:", scores.mean())
print("Std:", scores.std())

## 10. Model Comparision

In [ ]:
results = pd.DataFrame({
    "Model": ["Logistic Regression",
              "Random Forest"
    ],
    "AUC_PR": [
        auc_pr_lr,
        auc_pr_rf
    ],
    "F1": [
        f1_lr,
        f1_rf
    ]

})

results

## 11. Final Model Selection

Random Forest achieved higher AUC-PR and F1-score than Logistic Regression. Therefore, Random Forest was selected as the final model for fraud detection.

In [ ]:
import joblib

joblib.dump(best_rf, "../models/random_forest_model.pkl")